# Google Jobs scraper: prototype (Find work, step 3)

For each role × location × years-of-experience, this notebook:

1. Opens Google and searches `"<role> in <location> with <N> years of experience jobs"`.
2. Clicks the **Jobs** tab, or opens the Jobs view (`udm=8`) directly if the tab isn't found.
3. Scrolls the list **5 × 220 px** to load more cards.
4. Reads every card: title, company, location, "via" source, posted date, employment type, apply links and full description.

Years come from the parsed resume, searched at **floor and ceil**: 3.5 searches for 3 and 4, and 3.0 searches for 3 only.

**Run this locally** (your Windows machine and normal Chrome), not on Render.
Google doesn't allow automated searches. Expect a CAPTCHA now and then. The window stays visible so you can solve it, and the scraper waits for you.

**Setup, once:**
```
pip install playwright beautifulsoup4 pandas nest_asyncio
playwright install chromium
```

In [ ]:
# ---- Inputs: in the app these come from Job preferences + the parsed resume ----
ROLES = ["Data Analyst"]
LOCATIONS = ["Bengaluru"]
RESUME_YEARS = 4.5            # from resume parsing; None = leave experience out of the query

# ---- Scraper knobs ----
SCROLL_TIMES = 5
SCROLL_PX = 220
SCROLL_PAUSE_S = 1.5
QUERY_DELAY_S = (6, 12)       # random pause between Google searches (be gentle)
CLICK_FOR_MISSING = True      # click cards whose description isn't already in the page
MAX_CLICKS_PER_QUERY = 40
HEADLESS = False              # keep False: headless gets CAPTCHA'd much faster
USE_SYSTEM_CHROME = True      # use your installed Chrome (looks less like a bot)
PROFILE_DIR = ".gjobs_profile"  # cookies persist here, so you solve a CAPTCHA once, not every run
CAPTCHA_WAIT_S = 180
SAVE_DEBUG_HTML = False       # True = save each results page, so you can send it to me if a field comes back empty

## Selectors
Google's class names are obfuscated and change every few months. Keep them all in one place.
When the scraper suddenly returns 0 jobs, open the page, inspect a card and update this dict.

In [ ]:
SEL = {
    "title":      "div.tNxQIb",                 # Senior Analyst, Digital Analytics
    "company":    "div.a3jPc",                  # div.wHYlTd.MKCbgd.a3jPc -> Labcorp
    "loc_via":    "div.FqK3wc",                 # "Bengaluru, Karnataka • via Labcorp Careers"
    "chip":       "span.Yf9oye[aria-label]",    # "Posted 1 day ago", "Employment type Full–time"
    "click":      "div[jsname='OCDbef']",       # the clickable part of a card
    "desc_head":  "span.OOyDTc",                # first part of the description
    "desc_tail":  "span[jsname='ij8cu']",       # hidden "Show full description" remainder
    "apply":      "a.brKmxb",                   # "Apply on LinkedIn", "Apply directly on Labcorp Careers"
    "apply_panel": "div.NmXWu",                 # the apply-buttons strip in the open job panel
    "detail_chip": "span.fLsjxc",               # detail panel chips: "No Degree Mentioned", salary...
}

In [ ]:
import asyncio, json, math, os, random, re, sys, time
from datetime import datetime, timedelta, timezone
from urllib.parse import quote_plus, urlparse, parse_qsl, urlencode, urlunparse

import pandas as pd
from bs4 import BeautifulSoup
from bs4.element import NavigableString, TemplateString


def experience_variants(years):
    """3.5 -> [3, 4]; 3.0 -> [3]; 0.4 -> [0, 1]; None -> [None]."""
    if years is None:
        return [None]
    y = max(0.0, float(years))
    return sorted({math.floor(y), math.ceil(y)})


def build_query(role, location, years):
    role, location = role.strip(), location.strip()
    where = f" in {location}" if location else ""
    if years is None:
        return f"{role}{where} jobs"
    if years == 0:
        return f"{role}{where} for freshers jobs"
    unit = "year" if years == 1 else "years"
    return f"{role}{where} with {years} {unit} of experience jobs"


def plan_queries(roles, locations, years):
    out = []
    for r in roles:
        for l in locations:
            for y in experience_variants(years):
                out.append({"role": r, "location": l, "years": y, "query": build_query(r, l, y)})
    return out


for q in plan_queries(ROLES, LOCATIONS, RESUME_YEARS):
    print(q["query"])

## Parsing (pure HTML → rows)
No browser needed here, so you can test it against a saved page (see the self-test cell).
Google embeds each card's full details (description and apply links) in a `<template>` next to the card,
so most jobs need no click at all.

In [ ]:
_UTM = re.compile(r"^(utm_|ved$|sa$|usg$)")
_SPACE = re.compile(r"\s+")


# Google keeps card details inside <template>; BeautifulSoup skips those strings unless asked.
_TEXT_TYPES = (NavigableString, TemplateString)


def _txt(el):
    return el.get_text(types=_TEXT_TYPES) if el is not None else ""


def _clean(s):
    return _SPACE.sub(" ", (s or "").replace("\xa0", " ")).strip()


def _strip_tracking(url):
    p = urlparse(url)
    q = [(k, v) for k, v in parse_qsl(p.query, keep_blank_values=True) if not _UTM.match(k)]
    return urlunparse(p._replace(query=urlencode(q), fragment=""))


def _block_text(el):
    """Text with <br> kept as line breaks."""
    if el is None:
        return ""
    for br in el.find_all("br"):
        br.replace_with("\n")
    lines = [_clean(x) for x in _txt(el).split("\n")]
    return "\n".join(x for x in lines if x)


def posted_to_date(text, now=None):
    """'1 day ago' / '3 hours ago' / '2 weeks ago' / 'Just posted' -> UTC datetime (approximate)."""
    now = now or datetime.now(timezone.utc)
    t = (text or "").lower()
    if not t:
        return None
    if "just" in t or "today" in t:
        return now
    if "yesterday" in t:
        return now - timedelta(days=1)
    m = re.search(r"(\d+)\+?\s*(minute|min|hour|hr|day|week|month)", t)
    if not m:
        return None
    n, unit = int(m.group(1)), m.group(2)
    days = {"minute": n / 1440, "min": n / 1440, "hour": n / 24, "hr": n / 24, "day": n, "week": 7 * n, "month": 30 * n}[unit]
    return now - timedelta(days=days)


def _real_url(href, ping=None):
    """Apply anchors carry the real link in href; sometimes it is a /url?url=... redirect instead."""
    for raw in (href, ping):
        if not raw:
            continue
        if raw.startswith("http") and "google." not in urlparse(raw).netloc:
            return _strip_tracking(raw)
        q = dict(parse_qsl(urlparse(raw).query))
        for k in ("url", "q"):
            if q.get(k, "").startswith("http"):
                return _strip_tracking(q[k])
    return None


def apply_links_from(anchors):
    """anchors: dicts with title/text/href/ping (from BeautifulSoup or from the live page)."""
    out = []
    for a in anchors:
        label = _clean(a.get("title") or a.get("text") or "")
        if label and not label.lower().startswith("apply"):
            continue
        url = _real_url(a.get("href"), a.get("ping"))
        if url and url not in {x["url"] for x in out}:
            site = re.sub(r"^apply (directly )?on\s+", "", label, flags=re.I) or urlparse(url).netloc
            out.append({"site": site, "url": url, "direct": "directly" in label.lower()})
    return out


def _maindata_links(root):
    """Backup: Google also lists the apply links as jobs_posting_urls_N inside a data-maindata blob."""
    blob = " ".join(el.get("data-maindata", "") for el in root.select("[data-maindata]"))
    urls = re.findall(r'jobs_posting_urls_\d+","(https?://[^"]+)"', blob.replace("\\u003d", "=").replace("\\u0026", "&"))
    return [{"href": u} for u in urls]


def _details(soup, docid):
    """Description + apply links + detail chips for one job, from its <template> or the open detail panel."""
    out = {"description": "", "apply_links": [], "detail_chips": []}
    if not docid:
        return out
    for root in soup.select(f'[data-encoded-docid="{docid}"]'):
        head = root.select_one(SEL["desc_head"])
        if head and not out["description"]:
            tail = root.select_one(SEL["desc_tail"])
            out["description"] = (_block_text(head) + _block_text(tail)).strip()
        anchors = [{"title": a.get("title"), "text": _txt(a), "href": a.get("href"), "ping": a.get("ping")}
                   for a in root.select(SEL["apply"])] + _maindata_links(root)
        for link in apply_links_from(anchors):
            if link["url"] not in {x["url"] for x in out["apply_links"]}:
                out["apply_links"].append(link)
        if not out["detail_chips"]:
            out["detail_chips"] = [_clean(_txt(c)) for c in root.select(SEL["detail_chip"]) if _clean(_txt(c))]
    return out


def _card_root(title_el):
    """Walk up from the title to the card container that carries the job id."""
    el = title_el
    while el is not None and el.name:
        if el.get("data-preview-id") or (el.get("id", "").endswith("=") and el.get("jsname") == "y1Aese"):
            return el
        el = el.parent
    return title_el.find_parent("div", class_="EimVGf") or title_el.parent


def parse_jobs_html(html, query="", now=None):
    soup = BeautifulSoup(html, "html.parser")
    rows, seen = [], set()
    for t in soup.select(SEL["title"]):
        if t.find_parent("template"):          # titles inside templates are copies
            continue
        card = _card_root(t)
        docid = card.get("id") if card.get("id", "").endswith("=") else None
        if not docid:
            m = re.search(r"htidocid=([^&#]+)", (t.find_parent(attrs={"data-share-url": True}) or {}).get("data-share-url", ""))
            docid = m.group(1).replace("%3D", "=") if m else None
        key = docid or _clean(_txt(t))
        if key in seen:
            continue
        seen.add(key)

        lv = card.select_one(SEL["loc_via"])
        loc_via = _clean(_txt(lv)) if lv else ""
        parts = [p.strip() for p in loc_via.split("•")]
        location = parts[0] if parts else ""
        via = re.sub(r"^via\s+", "", parts[1], flags=re.I) if len(parts) > 1 else ""

        chips = {}
        for c in card.select(SEL["chip"]):
            lab = _clean(c["aria-label"])
            if lab.lower().startswith("posted"):
                chips["posted"] = _clean(_txt(c))
            elif lab.lower().startswith("employment type"):
                chips["employment_type"] = _clean(_txt(c))
            elif lab.lower().startswith("salary"):
                chips["salary"] = _clean(_txt(c))
            else:
                chips.setdefault("other", []).append(lab)

        d = _details(soup, docid)
        # Google lists the most relevant source first. "Apply directly on" isn't safe to prefer:
        # aggregators sometimes attach a different job under that label.
        primary = d["apply_links"][0]["url"] if d["apply_links"] else None
        posted_at = posted_to_date(chips.get("posted"), now)
        share = (t.find_parent(attrs={"data-share-url": True}) or {}).get("data-share-url")

        rows.append({
            "provider": "google_jobs",
            "external_id": docid,
            "title": _clean(_txt(t)),
            "company": _clean(_txt(card.select_one(SEL["company"]))) if card.select_one(SEL["company"]) else None,
            "location": location or None,
            "via": via or None,
            "remote": bool(re.search(r"\b(remote|work from home|anywhere)\b", f"{_txt(t)} {location}", re.I)),
            "employment_type": chips.get("employment_type"),
            "salary": chips.get("salary"),
            "posted_text": chips.get("posted"),
            "posted_at": posted_at.isoformat() if posted_at else None,
            "apply_url": primary,
            "apply_links": d["apply_links"],
            "detail_chips": d["detail_chips"],
            "description": d["description"],
            "google_url": share,
            "query": query,
        })
    return rows


def missing_details(rows):
    """Jobs still lacking a description or an apply link (these get clicked)."""
    return [r["external_id"] for r in rows if r["external_id"] and (not r["description"] or not r["apply_url"])]


def merge_panel(rows, panel):
    """panel: {docid: {"apply_links": [...], "description": str}} read live after clicking each card."""
    for r in rows:
        p = panel.get(r["external_id"])
        if not p:
            continue
        if p["apply_links"] and not r["apply_links"]:
            r["apply_links"] = p["apply_links"]
        if r["apply_links"] and not r["apply_url"]:
            r["apply_url"] = r["apply_links"][0]["url"]
        if p["description"] and not r["description"]:
            r["description"] = p["description"]
    return rows

## Self-test against the HTML you saved from Chrome (optional)
Save a card's outer HTML (or the whole page: Ctrl+S) to a file, and point `SAMPLE_HTML` at it.

In [ ]:
SAMPLE_HTML = "samples/gjobs_sample.html"   # two real cards saved from Chrome; swap in your own page (Ctrl+S)
if SAMPLE_HTML and os.path.exists(SAMPLE_HTML):
    with open(SAMPLE_HTML, encoding="utf-8") as f:
        sample = parse_jobs_html(f.read(), "sample")
    print(len(sample), "job(s) parsed")
    display(pd.DataFrame(sample)[["title", "company", "location", "via", "posted_text", "employment_type", "apply_url"]])
    print(sample[0]["description"][:600] if sample else "")

## Browser part (Playwright)
On Windows, Jupyter's event loop can't start the browser process, so the sync Playwright API runs in its own worker thread.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright, TimeoutError as PWTimeout


def _launch(p):
    kw = dict(user_data_dir=PROFILE_DIR, headless=HEADLESS, locale="en-IN",
              viewport={"width": 1366, "height": 900}, args=["--disable-blink-features=AutomationControlled"])
    if USE_SYSTEM_CHROME:
        try:
            return p.chromium.launch_persistent_context(channel="chrome", **kw)
        except Exception as e:
            print("System Chrome not found, using Playwright's Chromium:", str(e).splitlines()[0])
    return p.chromium.launch_persistent_context(**kw)


def _is_blocked(page):
    return "/sorry/" in page.url or page.locator("#captcha-form, form#captcha").count() > 0


def _wait_if_blocked(page):
    if not _is_blocked(page):
        return True
    print(f"Google is showing a CAPTCHA. Solve it in the browser window (waiting up to {CAPTCHA_WAIT_S}s)...")
    end = time.time() + CAPTCHA_WAIT_S
    while time.time() < end:
        time.sleep(2)
        if not _is_blocked(page):
            print("Thanks, carrying on.")
            return True
    print("Still blocked. Stopping this query.")
    return False


def _accept_consent(page):
    for name in ("Accept all", "I agree", "Reject all"):
        b = page.get_by_role("button", name=name)
        if b.count():
            b.first.click()
            page.wait_for_load_state("domcontentloaded")
            return


def _open_jobs_tab(page, query):
    page.goto(f"https://www.google.com/search?q={quote_plus(query)}&hl=en", wait_until="domcontentloaded")
    _accept_consent(page)
    if not _wait_if_blocked(page):
        return False
    tab = page.locator('div[role="listitem"] a:has(span:text-is("Jobs"))')
    if not tab.count():
        tab = page.get_by_role("link", name="Jobs", exact=True)
    try:
        if tab.count():
            tab.first.click()
            page.wait_for_selector(SEL["title"], timeout=12000)
            return True
    except PWTimeout:
        pass
    # Fallback: the Jobs view directly (this is the link Google itself shares)
    page.goto(f"https://www.google.com/search?q={quote_plus(query)}&ibp=htl;jobs&udm=8&hl=en", wait_until="domcontentloaded")
    if not _wait_if_blocked(page):
        return False
    try:
        page.wait_for_selector(SEL["title"], timeout=12000)
        return True
    except PWTimeout:
        print("No job cards found for:", query)
        return False


def _scroll(page):
    first = page.locator(SEL["title"]).first
    try:
        first.hover(timeout=3000)          # so the wheel scrolls the job list, not some other pane
    except PWTimeout:
        pass
    for i in range(SCROLL_TIMES):
        page.mouse.wheel(0, SCROLL_PX)
        page.evaluate(f"window.scrollBy(0, {SCROLL_PX})")
        page.wait_for_timeout(int(SCROLL_PAUSE_S * 1000))


_PANEL_JS = """
([docid, applySel, panelSel, headSel, tailSel]) => {
  const scoped = [...document.querySelectorAll(`[data-encoded-docid="${docid}"] ${applySel}`)];
  // Fallback: the apply strip of whichever job panel is open right now (visible on screen)
  const open = [...document.querySelectorAll(`${panelSel} ${applySel}`)].filter(a => a.offsetParent !== null);
  const anchors = (scoped.length ? scoped : open).map(a => ({
    title: a.getAttribute("title"), text: a.innerText, href: a.getAttribute("href"), ping: a.getAttribute("ping")}));
  const root = document.querySelector(`[data-encoded-docid="${docid}"] ${headSel}`)?.parentElement;
  const desc = root ? [root.querySelector(headSel), root.querySelector(tailSel)]
      .filter(Boolean).map(e => e.innerText || e.textContent).join("") : "";
  return {anchors, desc};
}
"""


def _fill_missing(page, rows):
    """Click cards that lack a description or an apply link, and read the open panel right away
    (Google re-uses one panel, so it has to be read before the next click replaces it)."""
    panel = {}
    for docid in missing_details(rows)[:MAX_CLICKS_PER_QUERY]:
        card = page.locator(f'[id="{docid}"] {SEL["click"]}')
        if not card.count():
            continue
        try:
            card.first.scroll_into_view_if_needed(timeout=3000)
            card.first.click(timeout=4000)
            page.wait_for_selector(f'{SEL["apply_panel"]} {SEL["apply"]}', state="visible", timeout=6000)
        except PWTimeout:
            continue
        page.wait_for_timeout(random.randint(500, 900))
        got = page.evaluate(_PANEL_JS, [docid, SEL["apply"], SEL["apply_panel"], SEL["desc_head"], SEL["desc_tail"]])
        panel[docid] = {"apply_links": apply_links_from(got["anchors"]), "description": _clean(got["desc"])}
    return panel


def scrape_query(page, q):
    t0 = time.time()
    if not _open_jobs_tab(page, q["query"]):
        return []
    _scroll(page)
    rows = parse_jobs_html(page.content(), q["query"])
    if SAVE_DEBUG_HTML:
        with open(f"debug_{re.sub(r'[^a-z0-9]+', '_', q['query'].lower())}.html", "w", encoding="utf-8") as f:
            f.write(page.content())
    if CLICK_FOR_MISSING and missing_details(rows):
        panel = _fill_missing(page, rows)
        rows = merge_panel(parse_jobs_html(page.content(), q["query"]), panel)
        print(f"  clicked {len(panel)} card(s) for details")
    for r in rows:
        r.update(role=q["role"], search_location=q["location"], search_years=q["years"])
    no_desc = sum(1 for r in rows if not r["description"])
    no_url = sum(1 for r in rows if not r["apply_url"])
    print(f"  {len(rows)} jobs in {time.time() - t0:.1f}s · missing description: {no_desc} · missing apply_url: {no_url}")
    return rows


def run_scrape(roles, locations, years):
    queries = plan_queries(roles, locations, years)
    out = []
    with sync_playwright() as p:
        ctx = _launch(p)
        page = ctx.pages[0] if ctx.pages else ctx.new_page()
        try:
            for i, q in enumerate(queries):
                print(f"[{i + 1}/{len(queries)}] {q['query']}")
                out.extend(scrape_query(page, q))
                if i < len(queries) - 1:
                    time.sleep(random.uniform(*QUERY_DELAY_S))
        finally:
            ctx.close()
    return out


def run_in_thread(fn, *args):
    def target():
        if sys.platform == "win32":
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        return fn(*args)
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(target).result()

## Run it

In [ ]:
raw = run_in_thread(run_scrape, ROLES, LOCATIONS, RESUME_YEARS)
print(len(raw), "rows before de-duplication")

## De-duplicate
The same post shows up for 4 and 5 years, and often under several sources.
Keys, in order: Google's job id, then the apply link without tracking, then title + company + city.

In [ ]:
def _norm_words(s):
    return re.sub(r"[^a-z0-9]+", " ", (s or "").lower()).strip()


def dedupe(rows):
    seen, out = set(), []
    for r in rows:
        city = (r.get("location") or "").split(",")[0]
        keys = [k for k in (r.get("external_id"), r.get("apply_url"),
                            f"{_norm_words(r['title'])}|{_norm_words(r.get('company'))}|{_norm_words(city)}") if k]
        if any(k in seen for k in keys):
            continue
        seen.update(keys)
        out.append(r)
    return out


jobs = dedupe(raw)
df = pd.DataFrame(jobs)
print(len(jobs), "unique jobs")
if len(df):
    display(df[["title", "company", "location", "via", "posted_text", "employment_type", "apply_url"]].head(30))

## Quality check
How much of each field came through. If `description` is low, raise `MAX_CLICKS_PER_QUERY` or check the selectors.

In [ ]:
if len(df):
    cols = ["title", "company", "location", "via", "posted_at", "employment_type", "apply_url", "description"]
    filled = {c: f"{(df[c].fillna('').astype(str).str.len() > 0).mean() * 100:.0f}%" for c in cols}
    print(json.dumps(filled, indent=2))
    print("\nBy source:\n", df["via"].value_counts().head(10).to_string())

## Preview: what the enrichment step will add (skills, years, degree)
This uses Hiriq's existing rules reader (`app/services/postingread.py`) if the backend is importable.
The real step (new `external_jobs` columns, NLP/transformer/GenAI parsing) comes next. This only shows what the text gives us.

In [ ]:
BACKEND = os.path.abspath(os.path.join("..", "..", "backend"))
try:
    sys.path.insert(0, BACKEND)
    from app.services import postingread
    if len(df):
        df["read"] = [postingread.read(t, d) for t, d in zip(df["title"], df["description"])]
        display(df[["title", "company", "read"]].head(15))
except Exception as e:
    print("Hiriq backend not importable from here, skipping:", e)

## Save
Shaped close to the `external_jobs` table so the adapter can insert it directly later.

In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M")
if len(df):
    df.to_json(f"gjobs_{stamp}.json", orient="records", force_ascii=False, indent=2)
    df.drop(columns=["apply_links", "detail_chips"]).to_csv(f"gjobs_{stamp}.csv", index=False, encoding="utf-8-sig")
    print("saved", f"gjobs_{stamp}.json / .csv")